In [ ]:
import os   
import sys  



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
from rdkit import Chem
from rdkit.Chem import AllChem  
import os
from torch_geometric.data import Data, DataLoader
import pandas as pd
import numpy as np
import numpy as np
from collections import Counter
import re  
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from matplotlib.ticker import ScalarFormatter
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import pandas as pd
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import LabelEncoder
from matplotlib.colors import ListedColormap
import seaborn as sns

import matplotlib.pyplot as plt
import seaborn as sns
import torch
from rdkit import Chem
import os
import pandas as pd
from torch_geometric.data import Data
import pandas as pd
from torch_geometric.data import DataLoader
from sklearn.model_selection import train_test_split
import os
from typing import List
import pandas as pd
from openbabel import openbabel, pybel
from utils import (get_node_features,
                   get_edge_index,
                   smiles_to_morgan_fingerprint,
                   cap_smiles,
                   add_hydrogens_to_smiles,
                   create_total_train_val_test_dataset,
                   save_data_list,
                   generate_lowest_energy_conformer_openbabel)

In [ ]:
from rdkit import RDLogger


RDLogger.DisableLog('rdApp.*')

In [ ]:
def extract_features_targets(data_list):
    
    X = []

    for data in data_list:
        fp = data.morgan_fp.numpy().flatten() 
        X.append(fp)
            
    X = np.array(X)
    return X

In [ ]:

import pandas as pd
import joblib
from typing import List

def predict_polymer_properties(
    df_polymer: pd.DataFrame,  
    model_to_predict_property_list: List[str]  
) -> pd.DataFrame:
    
    import gc  

    
    df_new = df_polymer.copy()  

    
    model_dict = {}  
    for prop in model_to_predict_property_list:  
        model_path = f"/path/to/example/polymer/static/model/{prop}_xgb_model.joblib"  
        model = joblib.load(model_path)  
        
        try:
            
            if hasattr(model, "set_params"):  
                try:
                    model.set_params(n_jobs=1)  
                except Exception:
                    pass  
            
            booster = getattr(model, "get_booster", lambda: None)()  
            if booster is not None and hasattr(booster, "set_param"):  
                try:
                    booster.set_param({"nthread": 1})  
                except Exception:
                    pass  
        except Exception as e:
            print("Public message removed for release.")  
        
        model_dict[prop] = model  

    
    for idx, row in df_new.iterrows():  
        psmiles = row.get('PSMILES')  
        try:
            
            psmiles_with_h = add_hydrogens_to_smiles(psmiles)  
            
            caped_smiles = cap_smiles(psmiles_with_h)  
        except Exception as e:
            
            print(f"Row {idx}: failed to process SMILES '{psmiles}' - {e}")  
            continue  

        try:
            
            morgan_fp_tensor = smiles_to_morgan_fingerprint(  
                caped_smiles, radius=2, n_bits=2048  
            )
            X = morgan_fp_tensor.numpy().reshape(1, -1)  
        except Exception as e:
            
            print(f"Row {idx}: failed to fingerprint SMILES '{caped_smiles}' - {e}")  
            continue  

        
        for prop, model in model_dict.items():  
            try:
                y_pred = model.predict(X)  
                
                df_new.at[idx, f"{prop}_pred"] = y_pred[0]  
            except Exception as e:
                
                print(f"Row {idx}: failed to predict '{prop}' for SMILES '{caped_smiles}' - {e}")  

    
    try:
        for k, m in list(model_dict.items()):  
            model_dict[k] = None  
            del m  
    except Exception:
        pass  
    try:
        del X  
    except Exception:
        pass  
    gc.collect()  

    return df_new  

In [ ]:
df_polymer_predict = pd.read_excel("polymer_list.xlsx")
selected_target_property_col_list = ['Tg', 'Dielectric_Constant_Total', 'Youngs_Modulus', 'Tm', 'Tensile_Strength']

df_polymer_predict = predict_polymer_properties(
                                    df_polymer_predict,
                                    selected_target_property_col_list
                                    )

df_polymer_predict.to_excel("polymer_list.xlsx", index=None)

In [ ]:
df_polymer_predict